# Streaming responses from Anthropic Claude models

This example demonstrates how to stream the response of an Anthropic Claude model from a Jupyter notebook. Instead of waiting for the complete answer, the notebook prints each fragment as the model produces it and reports the time to first token.

## 1. Setup

Set your Anthropic API key in the `ANTHROPIC_API_KEY` environment variable before running the notebook.

In [ ]:
%pip install anthropic

In [ ]:
from anthropic import Anthropic
import time

client = Anthropic()  # ANTHROPIC_API_KEY should be set as an environment variable


def query_model(prompt: str,
                model: str = "claude-haiku-4-5",
                max_tokens: int = 2048,
                temperature: float = 0) -> str:
    """Stream the response of an Anthropic model."""
    start = time.perf_counter()
    first_token = None
    chunks = []

    with client.messages.stream(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[
            {"role": "user", "content": prompt}
        ],
    ) as stream:
        for text in stream.text_stream:
            if first_token is None:
                first_token = time.perf_counter() - start
            print(text, end="", flush=True)
            chunks.append(text)
        message = stream.get_final_message()
    latency = time.perf_counter() - start
    print()

    usage = message.usage
    ttft = first_token if first_token is not None else latency
    print(f"\tModel: {message.model}")
    print(f"\tTime to first token: {ttft:.3f} seconds")
    print(f"\tLatency: {latency:.3f} seconds")
    print(f"\tInput tokens: {usage.input_tokens}")
    print(f"\tOutput tokens: {usage.output_tokens}")
    print(f"\tTotal tokens: {usage.input_tokens + usage.output_tokens}")

    return "".join(chunks).strip()

In [ ]:
prompt = "How many tokens are in your context window?"
print("User:", prompt)
print("Claude: ", end="", flush=True)
query_model(prompt)